In [ ]:
import pandas as pd

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import torch
from torch.utils.data import Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/V3_FIN/v3_all_news_senti_dateparsed.csv',
    usecols=["content", "industry"]
)
test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/V3_FIN/naver_news_250710_cleaned.csv')

In [ ]:
label_encoder = LabelEncoder()
train['label'] = label_encoder.fit_transform(train['industry'])
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60257 entries, 0 to 60256
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   industry  60257 non-null  object
 1   content   60256 non-null  object
 2   label     60257 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.4+ MB


In [ ]:
train.head(2)

,industry,content,label
0,정보통신업,유럽 최대 자동화 부품 특화 전시회 올해로 30주년 맞아 5G 클라우드 빅데이터 기...,12
1,조선업,우리회계법인 고동호 대표 KOTRA 글로벌 지역전문가 방콕무역관 자문위원 태국은 최...,13


In [ ]:
tokenizer = BertTokenizer.from_pretrained("monologg/kobert")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/263 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'KoBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]
        item = tokenizer(text, padding="max_length", truncation=True, max_length=512, return_tensors='pt')

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return {k: v.squeeze() for k, v in item.items()}


In [ ]:
train_dataset = NewsDataset(train['content'].tolist(), train['label'].tolist())
test_dataset = NewsDataset(test['content'].astype(str).tolist())

In [ ]:
model = BertForSequenceClassification.from_pretrained("monologg/kobert", num_labels=len(label_encoder.classes_))

config.json:   0%|          | 0.00/426 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/kobert_output",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=2e-5
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)
trainer.train()

/tmp/ipython-input-19-1564289777.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Exception in thread Thread-19 (_loader_worker):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch_xla/distributed/parallel_loader.py", line 165, in _loader_worker
    _, data = next(data_iter)
              ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/accelerate/data_loader.py", line 578, in __iter__
    next_batch = next(dataloader_iter)
                 ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 708, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr

Step,Training Loss
50,0.885900
100,0.887900
150,0.885600
200,0.973200
250,0.922700
300,0.912100
350,0.953800
400,0.861400
450,0.844200
500,0.913000


Exception in thread Thread-21 (_loader_worker):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch_xla/distributed/parallel_loader.py", line 165, in _loader_worker
    _, data = next(data_iter)
              ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/accelerate/data_loader.py", line 578, in __iter__
    next_batch = next(dataloader_iter)
                 ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 708, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 764, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^

TrainOutput(global_step=3505, training_loss=0.7996368753757014, metrics={'train_runtime': 740.6334, 'train_samples_per_second': 244.137, 'train_steps_per_second': 15.259, 'total_flos': 1.475699024535552e+16, 'train_loss': 0.7996368753757014, 'epoch': 2.5330501725511017})

In [ ]:
from tqdm import tqdm
import numpy as np
tqdm.pandas()

pred_output = trainer.predict(test_dataset)
probs = torch.nn.functional.softmax(torch.tensor(pred_output.predictions), dim=1).numpy()
pred_labels = np.argmax(probs, axis=1)
confidences = np.max(probs, axis=1)

In [ ]:
final_preds = []
threshold = 0.3

for label, conf in zip(pred_labels, confidences):
    if conf >= threshold:
        final_preds.append(label_encoder.inverse_transform([label])[0])
    else:
        final_preds.append(np.nan)

test['predicted_industry'] = final_preds


In [ ]:
test

,cat,date,title,content,url,domain,predicted_industry
0,101,2025.07.09. 오후 12:32,집값 오르니 주택연금 가입 넉 달 만에 감소,수도권을 중심으로 아파트 가격이 가파르게 오르면서 지난 5월 주택연금 신규 가입이 ...,https://n.news.naver.com/mnews/article/055/000...,n.news.naver.com,전자부품
1,101,2025.07.09. 오전 10:00,"삼성물산 ""개포우성7차 조합원 100% 열린조망·5베이"" 제안",10개 동 2열 주거동 배치 동간거리 최대 43m 확보개포지구 최고 높이 천장고 최...,https://n.news.naver.com/mnews/article/018/000...,n.news.naver.com,건설업
2,101,2025.07.08. 오후 2:29,"""전기차부터 UAM까지""…국제 e-모빌리티엑스포, 제주서 개최",e 모빌리티의 다보스포럼 제12회IEVE202550개국 150여개 국내외 기업 및 ...,https://n.news.naver.com/mnews/article/018/000...,n.news.naver.com,에너지
3,101,2025.07.09. 오전 10:06,"현대百, 중고 패션 보상 프로그램 '바이백' 도입",중고 패션 상품 되팔면 H포인트로 지급현대백화점은 중고 패션 보상 프로그램 바이백 ...,https://n.news.naver.com/mnews/article/031/000...,n.news.naver.com,NaN
4,101,2025.07.09. 오전 12:01,"전력 수요량 심상찮다, 폭염에 이틀째 올 최고…예년 같으면 7월말 수준",이른 무더위가 찾아오면서 전력 수요가 역대 7월 기준 최고치를 기록했다 정부가 본격...,https://n.news.naver.com/mnews/article/025/000...,n.news.naver.com,에너지
...,...,...,...,...,...,...,...
11740,104,2025.07.02. 오후 6:01,"트럼프 발언 수습 나선 이시바 ""日, 최대 대미 투자국...관세보다 국익""",이시바 관세율보다 투자가 중요 강조에도 언론 트럼프 적자 쌀 언급 상황 엄중 이시바...,https://n.news.naver.com/mnews/article/469/000...,n.news.naver.com,석유정제
11741,104,2025.07.02. 오후 6:01,"이-이 휴전 탄력 받은 트럼프 ""하마스, 가자 휴전안 수용하라""",SNS에 이스라엘은 동의 하마스 압박이스라엘 이란 휴전 중동중재 자신감 완파 주장 ...,https://n.news.naver.com/mnews/article/011/000...,n.news.naver.com,석유정제
11742,104,2025.07.02. 오후 5:59,"""협상 비협조"" 콕 집은 트럼프…日에 '35% 본보기 관세' 때리나",한국 등 합의 난항국에 경고 곧 관세 서한 일부는 무역 못할 수도 선제협상 불구 7...,https://n.news.naver.com/mnews/article/011/000...,n.news.naver.com,금속제조업
11743,104,2025.07.02. 오후 5:58,"美 해군 병력·기지 감시한 스파이?…법무부, 中 국적자 2명 기소 [핫이슈]",미국 오리건주에 거주하는 중국 국적자 위안스 첸 38 이 2023년 미 해군 항공모...,https://n.news.naver.com/mnews/article/081/000...,n.news.naver.com,NaN


In [ ]:

test['predicted_industry'].isna().sum()

np.int64(1347)

In [ ]:
test.to_csv('/content/drive/MyDrive/Colab Notebooks/V3_FIN/naver_news_250710_bert_fin.csv',index=False)